# 📊 Module 8: Model Evaluation & Cross-Validation
## Theory + Practical

---

## 📚 THEORY

### Why Evaluation Matters?
A model that scores 99% on training data might only score 60% on real new data — that's **overfitting**. Proper evaluation tells you how your model will perform in the **real world**.

### Cross-Validation (CV)
Instead of a single train/test split, use **K-Fold CV**:
```
Fold 1:  [TEST] [train] [train] [train] [train]  → Score 1
Fold 2:  [train] [TEST] [train] [train] [train]  → Score 2
Fold 3:  [train] [train] [TEST] [train] [train]  → Score 3
...
Final Score = Average of all fold scores
```
This gives a **more reliable** estimate of real performance.

### Regression Metrics
| Metric | Formula | Note |
|--------|---------|------|
| MAE | mean(|y - ŷ|) | Easy to interpret |
| RMSE | √mean((y-ŷ)²) | Penalizes large errors |
| R² | 1 - SS_res/SS_tot | 1.0 = perfect fit |

### Classification Metrics
| Metric | When to Use |
|--------|-------------|
| Accuracy | Balanced classes |
| Precision | When false positives are costly |
| Recall | When false negatives are costly |
| F1 | Imbalanced classes |
| ROC-AUC | Overall probability ranking |

---

## 🔬 PRACTICAL

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import cross_val_score, KFold, StratifiedKFold, learning_curve
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import warnings; warnings.filterwarnings('ignore')

print('Libraries loaded ✅')

### Practical 1: K-Fold Cross-Validation

In [ ]:
cancer = load_breast_cancer()
X, y = cancer.data, cancer.target

# Build a pipeline (scaling + model)
model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))

# 5-Fold cross-validation
cv_scores = cross_val_score(model, X, y, cv=5, scoring='accuracy')

print(f'CV Scores (each fold): {cv_scores.round(4)}')
print(f'Mean:  {cv_scores.mean():.4f}')
print(f'Std:   {cv_scores.std():.4f}  ← smaller is better (stable model)')

# Visualize fold scores
plt.figure(figsize=(7, 4))
plt.bar(range(1, 6), cv_scores, color='steelblue', edgecolor='black')
plt.axhline(cv_scores.mean(), color='red', linestyle='--', label=f'Mean={cv_scores.mean():.3f}')
plt.title('5-Fold Cross-Validation Scores')
plt.xlabel('Fold')
plt.ylabel('Accuracy')
plt.ylim(0.9, 1.0)
plt.legend()
plt.tight_layout()
plt.show()

### Practical 2: Learning Curves — Diagnosing Overfitting/Underfitting

In [ ]:
# Learning curve shows how model performance changes with more training data
train_sizes, train_scores, val_scores = learning_curve(
    model, X, y, cv=5, train_sizes=np.linspace(0.1, 1.0, 10), scoring='accuracy'
)

train_mean = train_scores.mean(axis=1)
val_mean   = val_scores.mean(axis=1)
train_std  = train_scores.std(axis=1)
val_std    = val_scores.std(axis=1)

plt.figure(figsize=(8, 5))
plt.plot(train_sizes, train_mean, 'o-', color='blue', label='Training score')
plt.plot(train_sizes, val_mean, 'o-', color='green', label='CV score')
plt.fill_between(train_sizes, train_mean - train_std, train_mean + train_std, alpha=0.15, color='blue')
plt.fill_between(train_sizes, val_mean - val_std, val_mean + val_std, alpha=0.15, color='green')
plt.title('Learning Curve')
plt.xlabel('Training Set Size')
plt.ylabel('Accuracy')
plt.legend()
plt.tight_layout()
plt.show()

gap = train_mean[-1] - val_mean[-1]
print(f'Train-Val gap at full data: {gap:.4f}')
if gap < 0.02:
    print('✅ Small gap → Good generalization')
else:
    print('⚠️  Large gap → Possible overfitting')

### Practical 3: Comparing Multiple Models with CV

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC

models = {
    'Logistic Regression': make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'Decision Tree': DecisionTreeClassifier(max_depth=5, random_state=42),
    'KNN (k=5)': make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5)),
    'SVM (RBF)': make_pipeline(StandardScaler(), SVC(kernel='rbf', random_state=42))
}

results = {}
for name, m in models.items():
    scores = cross_val_score(m, X, y, cv=5, scoring='f1')
    results[name] = scores
    print(f'{name:<25} F1: {scores.mean():.4f} ± {scores.std():.4f}')

# Box plot comparison
plt.figure(figsize=(9, 5))
plt.boxplot(results.values(), labels=results.keys(), patch_artist=True)
plt.xticks(rotation=20, ha='right')
plt.title('Model Comparison via 5-Fold CV (F1 Score)')
plt.ylabel('F1 Score')
plt.tight_layout()
plt.show()

### 🏋️ Try It Yourself!

In [ ]:
# ✏️ YOUR TURN: Change scoring metric and see which model wins
# Options: 'accuracy', 'f1', 'precision', 'recall', 'roc_auc'
metric = 'roc_auc'  # Change this!

for name, m in models.items():
    scores = cross_val_score(m, X, y, cv=5, scoring=metric)
    print(f'{name:<25} {metric}: {scores.mean():.4f}')

## 🎓 Summary

| Concept | Takeaway |
|---------|----------|
| Train/Test split | Quick check, but unreliable with small data |
| K-Fold CV | More reliable — uses all data for evaluation |
| Learning curve | Diagnose overfitting vs underfitting |
| Small train-val gap | Model generalizes well |
| Large gap | Model is overfitting — add regularization |
| Always use CV | Before choosing a final model |

➡️ **Next: Feature Engineering**